# Progetto NAML — PINN con DeepXDE su GPU (casi lunghi)

Notebook per i tre casi che richiedono ore su CPU:
**Allen–Cahn 1D**, **Schrödinger 1D**, **Schrödinger 2D**
(paper: Grossmann et al., *Can Physics-Informed Neural Networks beat the Finite Element Method?*, arXiv:2302.04107).

**Prima di eseguire:** `Runtime` → `Cambia tipo di runtime` → Acceleratore hardware: **GPU (T4)**.

Poi `Runtime` → `Esegui tutto`. I risultati (CSV, JSON, PNG) vengono scaricati come zip nell'ultima cella.
Gli script sono identici a quelli nella cartella del progetto: i CSV prodotti qui si confrontano
direttamente con la parte FEM.

In [ ]:
# Verifica GPU
!nvidia-smi

In [ ]:
# Installazione (PyTorch e' gia' presente su Colab)
%pip install -q deepxde

## Scrittura degli script (identici a quelli del progetto)

In [ ]:
%%writefile common.py
import argparse, json, os, time
import numpy as np
import deepxde as dde


def get_parser(description, default_arch, default_adam, default_lr):
    p = argparse.ArgumentParser(description=description,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--arch", type=int, nargs="+", default=default_arch)
    p.add_argument("--adam-iters", type=int, default=default_adam)
    p.add_argument("--lr", type=float, default=default_lr)
    p.add_argument("--lbfgs-iters", type=int, default=15000)
    p.add_argument("--no-lbfgs", action="store_true")
    p.add_argument("--fast", action="store_true")
    p.add_argument("--seed", type=int, default=1234)
    p.add_argument("--outdir", default="results")
    p.add_argument("--dist", default="Hammersley",
                   choices=["Hammersley", "LHS", "pseudo"])
    p.add_argument("--resample-period", type=int, default=100)
    return p


def apply_fast(args, adam=1000, lbfgs=200):
    if args.fast:
        args.adam_iters = min(args.adam_iters, adam)
        args.lbfgs_iters = min(args.lbfgs_iters, lbfgs)
    return args


def train_pinn(data, net, lr, adam_iters, loss_weights=None, pretrain=None,
               use_lbfgs=True, lbfgs_maxiter=15000, resample_period=100,
               metrics=None):
    model = dde.Model(data, net)
    callbacks = []
    if resample_period and resample_period > 0:
        callbacks.append(dde.callbacks.PDEPointResampler(period=resample_period))
    t0 = time.perf_counter()
    if pretrain is not None:
        it_pre, lr_pre, w_pre = pretrain
        model.compile("adam", lr=lr_pre, loss_weights=w_pre, metrics=metrics)
        model.train(iterations=it_pre, display_every=1000)
    model.compile("adam", lr=lr, loss_weights=loss_weights, metrics=metrics)
    losshistory, train_state = model.train(
        iterations=adam_iters, callbacks=callbacks, display_every=1000)
    if use_lbfgs and lbfgs_maxiter > 0:
        dde.optimizers.config.set_LBFGS_options(maxiter=lbfgs_maxiter)
        model.compile("L-BFGS", loss_weights=loss_weights, metrics=metrics)
        losshistory, train_state = model.train(display_every=1000)
    train_time = time.perf_counter() - t0
    return model, losshistory, train_state, train_time


def timed_predict(model, X, batch_size=100_000):
    t0 = time.perf_counter()
    parts = [model.predict(X[i:i + batch_size]) for i in range(0, len(X), batch_size)]
    y = np.vstack(parts)
    return y, time.perf_counter() - t0


def rel_l2(y_pred, y_true):
    return float(np.linalg.norm(y_pred - y_true) / np.linalg.norm(y_true))


def save_run(outdir, name, arch, times, errors, X, y_pred, columns, y_true=None,
             losshistory=None, train_state=None):
    os.makedirs(outdir, exist_ok=True)
    blocks = [X, y_pred]
    if y_true is not None:
        blocks.append(y_true)
    table = np.hstack(blocks)
    csv_path = os.path.join(outdir, f"{name}_pred.csv")
    np.savetxt(csv_path, table, delimiter=",", header=",".join(columns), comments="")
    info = {"case": name, "architecture": list(arch),
            "times_sec": times, "errors": errors}
    with open(os.path.join(outdir, f"{name}_info.json"), "w") as f:
        json.dump(info, f, indent=2)
    if losshistory is not None and train_state is not None:
        dde.utils.saveplot(losshistory, train_state, issave=True, isplot=False,
                           output_dir=outdir)
    print(f"\n=== {name} ===")
    print(json.dumps(info, indent=2))
    print(f"Predizione salvata in: {csv_path}")
    return csv_path


In [ ]:
%%writefile allen_cahn_1d.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import deepxde as dde
from common import (get_parser, apply_fast, train_pinn, timed_predict,
                    save_run)

EPS = 0.01
T_FINAL = 0.05


def init_cond(X):
    x = X[:, 0:1]
    return 0.25 * np.sin(2 * np.pi * x) + 0.25 * np.sin(16 * np.pi * x) + 0.5


def pde(X, u):
    u_t = dde.grad.jacobian(u, X, i=0, j=1)
    u_xx = dde.grad.hessian(u, X, i=0, j=0)
    return u_t - EPS * u_xx + (2.0 / EPS) * u * (1 - u) * (1 - 2 * u)


def main():
    parser = get_parser(__doc__, default_arch=[100, 100, 100, 100],
                        default_adam=50000, default_lr=1e-4)
    parser.add_argument("--pretrain-iters", type=int, default=7000)
    args = parser.parse_args()
    apply_fast(args, adam=2000, lbfgs=200)
    if args.fast:
        args.pretrain_iters = min(args.pretrain_iters, 500)
    dde.config.set_random_seed(args.seed)
    geom = dde.geometry.Interval(0, 1)
    timedomain = dde.geometry.TimeDomain(0, T_FINAL)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)
    bc = dde.icbc.PeriodicBC(geomtime, 0,
                             lambda X, on_boundary: on_boundary,
                             derivative_order=0, component=0)
    ic = dde.icbc.IC(geomtime, init_cond, lambda X, on_initial: on_initial)
    nf, ng, nh = (2000, 50, 100) if args.fast else (20000, 250, 500)
    data = dde.data.TimePDE(geomtime, pde, [bc, ic],
                            num_domain=nf, num_boundary=ng,
                            num_initial=nh, train_distribution=args.dist)
    net = dde.nn.FNN([2] + args.arch + [1], "tanh", "Glorot normal")
    weights_full = [1, 1, 1000]
    weights_ic_only = [0, 0, 1]
    model, losshistory, train_state, t_train = train_pinn(
        data, net, args.lr, args.adam_iters,
        loss_weights=weights_full,
        pretrain=(args.pretrain_iters, args.lr, weights_ic_only),
        use_lbfgs=not args.no_lbfgs, lbfgs_maxiter=args.lbfgs_iters,
        resample_period=args.resample_period)
    nx, nt = 512, 51
    xs = np.linspace(0, 1, nx)
    ts = np.linspace(0, T_FINAL, nt)
    XX, TT = np.meshgrid(xs, ts, indexing="ij")
    X = np.stack([XX.ravel(), TT.ravel()], axis=1)
    y_pred, t_eval = timed_predict(model, X)
    name = "allen_cahn_1d_" + "-".join(map(str, args.arch))
    save_run(args.outdir, name, args.arch,
             {"train": t_train, "eval": t_eval},
             {"nota": "errore da calcolare vs FEM"},
             X, y_pred, ["x", "t", "u_pinn"],
             losshistory=losshistory, train_state=train_state)
    U = y_pred.reshape(nx, nt)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    im = axes[0].pcolormesh(TT, XX, U, shading="auto")
    fig.colorbar(im, ax=axes[0])
    for j, lbl in [(0, "t=0"), (nt // 2, "t=T/2"), (nt - 1, "t=T")]:
        axes[1].plot(xs, U[:, j], label=lbl)
    axes[1].plot(xs, init_cond(xs[:, None]), "k:", label="cond. iniziale")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(f"{args.outdir}/{name}.png", dpi=150)
    print(f"Grafico salvato in: {args.outdir}/{name}.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile schrodinger_1d.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import deepxde as dde
from common import (get_parser, apply_fast, train_pinn, timed_predict,
                    save_run)

X_MIN, X_MAX = -5.0, 5.0
T_FINAL = np.pi / 2


def pde(X, h):
    u, v = h[:, 0:1], h[:, 1:2]
    u_t = dde.grad.jacobian(h, X, i=0, j=1)
    v_t = dde.grad.jacobian(h, X, i=1, j=1)
    u_xx = dde.grad.hessian(h, X, component=0, i=0, j=0)
    v_xx = dde.grad.hessian(h, X, component=1, i=0, j=0)
    mod2 = u ** 2 + v ** 2
    f_u = u_t + 0.5 * v_xx + mod2 * v
    f_v = v_t - 0.5 * u_xx - mod2 * u
    return [f_u, f_v]


def main():
    args = get_parser(__doc__, default_arch=[100, 100, 100, 100],
                      default_adam=50000, default_lr=1e-4).parse_args()
    apply_fast(args, adam=2000, lbfgs=200)
    dde.config.set_random_seed(args.seed)
    geom = dde.geometry.Interval(X_MIN, X_MAX)
    timedomain = dde.geometry.TimeDomain(0, T_FINAL)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)
    on_b = lambda X, on_boundary: on_boundary
    bcs = [
        dde.icbc.PeriodicBC(geomtime, 0, on_b, derivative_order=0, component=0),
        dde.icbc.PeriodicBC(geomtime, 0, on_b, derivative_order=0, component=1),
        dde.icbc.PeriodicBC(geomtime, 0, on_b, derivative_order=1, component=0),
        dde.icbc.PeriodicBC(geomtime, 0, on_b, derivative_order=1, component=1),
    ]
    on_init = lambda X, on_initial: on_initial
    ics = [
        dde.icbc.IC(geomtime, lambda X: 2 / np.cosh(X[:, 0:1]), on_init,
                    component=0),
        dde.icbc.IC(geomtime, lambda X: 0, on_init, component=1),
    ]
    nf = 2000 if args.fast else 20000
    data = dde.data.TimePDE(geomtime, pde, bcs + ics,
                            num_domain=nf, num_boundary=50,
                            num_initial=50, train_distribution=args.dist)
    net = dde.nn.FNN([2] + args.arch + [2], "tanh", "Glorot normal")
    model, losshistory, train_state, t_train = train_pinn(
        data, net, args.lr, args.adam_iters,
        use_lbfgs=not args.no_lbfgs, lbfgs_maxiter=args.lbfgs_iters,
        resample_period=args.resample_period)
    nx, nt = 256, 101
    xs = np.linspace(X_MIN, X_MAX, nx)
    ts = np.linspace(0, T_FINAL, nt)
    XX, TT = np.meshgrid(xs, ts, indexing="ij")
    X = np.stack([XX.ravel(), TT.ravel()], axis=1)
    y_pred, t_eval = timed_predict(model, X)
    u, v = y_pred[:, 0:1], y_pred[:, 1:2]
    mod = np.sqrt(u ** 2 + v ** 2)
    name = "schrodinger_1d_" + "-".join(map(str, args.arch))
    save_run(args.outdir, name, args.arch,
             {"train": t_train, "eval": t_eval},
             {"nota": "errore da calcolare vs FEM"},
             X, np.hstack([u, v, mod]),
             ["x", "t", "u_re", "u_im", "abs_h"],
             losshistory=losshistory, train_state=train_state)
    H = mod.reshape(nx, nt)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    im = axes[0].pcolormesh(TT, XX, H, shading="auto")
    fig.colorbar(im, ax=axes[0])
    for j, lbl in [(0, "t=0"), (nt // 2, "t=pi/4"), (nt - 1, "t=pi/2")]:
        axes[1].plot(xs, H[:, j], label=lbl)
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(f"{args.outdir}/{name}.png", dpi=150)
    print(f"Grafico salvato in: {args.outdir}/{name}.png")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile schrodinger_2d.py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import deepxde as dde
from common import (get_parser, apply_fast, train_pinn, timed_predict,
                    save_run)

L = 5.0
T_FINAL = np.pi / 2


def sech(z):
    return 1.0 / np.cosh(z)


def init_re(X):
    x, y = X[:, 0:1], X[:, 1:2]
    return sech(x) + 0.5 * sech(y - 2) + 0.5 * sech(y + 2)


def pde(X, h):
    u, v = h[:, 0:1], h[:, 1:2]
    u_t = dde.grad.jacobian(h, X, i=0, j=2)
    v_t = dde.grad.jacobian(h, X, i=1, j=2)
    lap_u = (dde.grad.hessian(h, X, component=0, i=0, j=0)
             + dde.grad.hessian(h, X, component=0, i=1, j=1))
    lap_v = (dde.grad.hessian(h, X, component=1, i=0, j=0)
             + dde.grad.hessian(h, X, component=1, i=1, j=1))
    mod2 = u ** 2 + v ** 2
    f_u = u_t + 0.5 * lap_v + mod2 * v
    f_v = v_t - 0.5 * lap_u - mod2 * u
    return [f_u, f_v]


def on_bnd_x(X, on_boundary):
    return on_boundary and np.isclose(abs(X[0]), L)


def on_bnd_y(X, on_boundary):
    return on_boundary and np.isclose(abs(X[1]), L)


def main():
    args = get_parser(__doc__, default_arch=[100, 100, 100, 100],
                      default_adam=50000, default_lr=1e-3).parse_args()
    apply_fast(args, adam=2000, lbfgs=200)
    dde.config.set_random_seed(args.seed)
    geom = dde.geometry.Rectangle([-L, -L], [L, L])
    timedomain = dde.geometry.TimeDomain(0, T_FINAL)
    geomtime = dde.geometry.GeometryXTime(geom, timedomain)
    bcs = []
    for comp_x, on_b in [(0, on_bnd_x), (1, on_bnd_y)]:
        for der in (0, 1):
            for comp in (0, 1):
                bcs.append(dde.icbc.PeriodicBC(
                    geomtime, comp_x, on_b,
                    derivative_order=der, component=comp))
    on_init = lambda X, on_initial: on_initial
    ics = [
        dde.icbc.IC(geomtime, init_re, on_init, component=0),
        dde.icbc.IC(geomtime, lambda X: 0, on_init, component=1),
    ]
    nf = 1000 if args.fast else 5000
    data = dde.data.TimePDE(geomtime, pde, bcs + ics,
                            num_domain=nf, num_boundary=100,
                            num_initial=100, train_distribution=args.dist)
    net = dde.nn.FNN([3] + args.arch + [2], "tanh", "Glorot normal")
    model, losshistory, train_state, t_train = train_pinn(
        data, net, args.lr, args.adam_iters,
        use_lbfgs=not args.no_lbfgs, lbfgs_maxiter=args.lbfgs_iters,
        resample_period=args.resample_period)
    n = 128
    xs = np.linspace(-L, L, n)
    XX, YY = np.meshgrid(xs, xs, indexing="ij")
    t_slice = np.pi / 4
    X = np.stack([XX.ravel(), YY.ravel(), np.full(n * n, t_slice)], axis=1)
    y_pred, t_eval = timed_predict(model, X)
    u, v = y_pred[:, 0:1], y_pred[:, 1:2]
    mod = np.sqrt(u ** 2 + v ** 2)
    name = "schrodinger_2d_" + "-".join(map(str, args.arch))
    save_run(args.outdir, name, args.arch,
             {"train": t_train, "eval": t_eval},
             {"nota": "errore da calcolare vs FEM; sezione a t=pi/4"},
             X, np.hstack([u, v, mod]),
             ["x", "y", "t", "u_re", "u_im", "abs_h"],
             losshistory=losshistory, train_state=train_state)
    H = mod.reshape(n, n)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    H0 = init_re(np.stack([XX.ravel(), YY.ravel()], axis=1)).reshape(n, n)
    im0 = axes[0].pcolormesh(XX, YY, np.abs(H0), shading="auto")
    fig.colorbar(im0, ax=axes[0])
    im1 = axes[1].pcolormesh(XX, YY, H, shading="auto")
    fig.colorbar(im1, ax=axes[1])
    fig.tight_layout()
    fig.savefig(f"{args.outdir}/{name}.png", dpi=150)
    print(f"Grafico salvato in: {args.outdir}/{name}.png")


if __name__ == "__main__":
    main()


## Allen–Cahn 1D — setup completo del paper
Pre-training 7000 it. sulla condizione iniziale + Adam 50000 it. (lr 1e-4, peso IC ×1000) + L-BFGS.
Su T4 indicativamente 20–40 min. Nel paper le reti da 20 nodi falliscono: usare ≥100 nodi per layer.

In [ ]:
!DDE_BACKEND=pytorch python allen_cahn_1d.py --arch 100 100 100 100 --outdir results

## Schrödinger 1D — setup completo del paper
Adam 50000 it., lr 1e-4, Nf=20000. Su T4 indicativamente 30–60 min.

In [ ]:
!DDE_BACKEND=pytorch python schrodinger_1d.py --arch 100 100 100 100 --outdir results

## Schrödinger 2D — setup completo del paper
Adam 50000 it., lr 1e-3, Nf=5000. Su T4 indicativamente 30–60 min.

In [ ]:
!DDE_BACKEND=pytorch python schrodinger_2d.py --arch 100 100 100 100 --outdir results

## (Opzionale) Altre architetture per le curve tempo-vs-errore
Decommenta e adatta: ogni run salva file distinti in `results/`.

In [ ]:
# for arch in ["20 20 20", "100 100 100", "100 100 100 100 100"]:
#     !DDE_BACKEND=pytorch python allen_cahn_1d.py --arch {arch} --outdir results
#     !DDE_BACKEND=pytorch python schrodinger_1d.py --arch {arch} --outdir results

## Download dei risultati

In [ ]:
!zip -r -q results_pinn_gpu.zip results
from google.colab import files
files.download("results_pinn_gpu.zip")